# Lesson 04: Classical machine learning

*Worked solutions for [the lesson page](../../lessons/04-classical-ml.html). The exercise version without solutions is `../04-classical-ml.ipynb`.*

The algorithms that still run much of the world, from linear and logistic regression built by hand to random forests, gradient boosting, clustering and PCA, finishing with an end-to-end scikit-learn project.

**Runs on:** Any computer (CPU). The cross-validation cells take a minute.

**How to use it:** read a section of the lesson, then run the matching cells here, in order. Change numbers, break things on purpose, and note what happens in your learning log. Cells that need an API key, a GPU or a download say so in their first line.

## Setup

Install what this lesson needs. Skip this cell if your environment from lesson 00 already has these packages.

In [ ]:
%pip install -q numpy pandas scikit-learn

## Why classical ML still matters

All the models here share scikit-learn's interface, so once you learn one you know them all:

*Shown for reference, not run here:*

```python
model = SomeModel(hyperparameters)
model.fit(X_train, y_train)          # learn parameters
y_pred = model.predict(X_test)       # use them
model.predict_proba(X_test)          # classifiers: class probabilities
```

## Linear and logistic regression

### Linear regression

$$
\boldsymbol{\theta} = (X^\top X)^{-1} X^\top \mathbf{y}
$$

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
x = rng.uniform(0, 10, 200)
y = 2.5 * x + 4 + rng.normal(0, 2, 200)

Xb = np.c_[np.ones_like(x), x]                     # add a column of ones for the intercept
theta = np.linalg.solve(Xb.T @ Xb, Xb.T @ y)       # solve rather than invert: faster and more stable
print(theta)                                       # [4.19 2.48]  intercept, slope

### Logistic regression: linear models for classification

The loss is binary cross-entropy (lesson 02), and its gradient turns out beautifully simple: the error times the input, exactly as in linear regression.

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

# two overlapping 2-D blobs
n = 400
X = np.vstack([rng.normal([-1, -1], 1.0, (n // 2, 2)),
               rng.normal([1.5, 1.0], 1.0, (n // 2, 2))])
y = np.r_[np.zeros(n // 2), np.ones(n // 2)]

w, b, lr = np.zeros(2), 0.0, 0.1
for step in range(1000):
    p = sigmoid(X @ w + b)             # (400,) predicted probabilities
    grad_w = X.T @ (p - y) / n         # (2,)
    grad_b = (p - y).mean()
    w -= lr * grad_w
    b -= lr * grad_b

print(w.round(2), round(b, 2), ((p > 0.5) == y).mean())   # about [2.28 1.70] -0.4 0.94

from sklearn.linear_model import LogisticRegression
sk = LogisticRegression(penalty=None).fit(X, y)
print(sk.coef_.round(2), sk.score(X, y))                  # about [[2.39 1.77]] 0.94

## Unsupervised learning

### k-means clustering

k-means finds $k$ cluster centres: assign each point to its nearest centre, move each centre to the mean of its points, repeat until nothing changes. You choose $k$. The "elbow" method plots the within-cluster distance (inertia) against $k$ and looks for where adding clusters stops helping much.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

digits = load_digits()                 # 1,797 handwritten digits as 8x8 = 64 pixel features
km = KMeans(n_clusters=10, n_init=10, random_state=0).fit(digits.data)
print(adjusted_rand_score(digits.target, km.labels_))   # about 0.67: clusters align with digits fairly well, no labels used

### PCA: compressing features

Principal component analysis finds the directions of greatest variance in the data (via the SVD from lesson 02) and projects onto the top few. Use it to visualise high-dimensional data in 2-D, to compress features, or to remove noise.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.9).fit(digits.data)
print(pca.n_components_)               # 21 of 64 dimensions keep 90% of the variance
Z = PCA(n_components=2).fit_transform(digits.data)   # (1797, 2): ready to scatter-plot, coloured by digit

### Anomaly detection

Many real problems have almost no labelled failures. Anomaly detectors learn what "normal" looks like and flag points that do not fit. **Isolation Forest** isolates points with random splits; anomalies are isolated in fewer splits because they are few and different.

In [ ]:
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(3)
normal = rng.normal(0, 1, (500, 3))
anomalies = rng.uniform(-6, 6, (15, 3))
X = np.vstack([normal, anomalies])

iso = IsolationForest(contamination=0.03, random_state=0).fit(X)
pred = iso.predict(X)                  # -1 = anomaly, 1 = normal
print((pred == -1).sum(), (pred[-15:] == -1).sum())   # flags 16 points, 14 of the 15 real anomalies

## End-to-end project: predicting instrument failures

Time to put it together on a realistic tabular problem: a fleet of field instruments, and we want to predict which will fail in the next 90 days so technicians can visit them first. We generate the data so you can run it anywhere, and so we know the truth it hides.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 3000
df = pd.DataFrame({
    "age_days": rng.integers(10, 2000, n),
    "site": rng.choice(["coastal", "urban", "desert", "alpine"], n, p=[0.3, 0.4, 0.2, 0.1]),
    "mean_temp_c": rng.normal(20, 8, n).round(1),
    "humidity_pct": rng.uniform(10, 95, n).round(1),
    "power_cycles": rng.poisson(30, n),
    "firmware": rng.choice(["v1", "v2", "v3"], n),
})
# the hidden truth: older, humid, coastal, often power-cycled units fail more;
# firmware v3 helps; temperatures far from 20 °C hurt
logit = (-4.5 + 0.0012 * df.age_days + 0.03 * (df.humidity_pct - 50) + 0.7 * (df.site == "coastal")
         + 0.04 * df.power_cycles - 0.9 * (df.firmware == "v3") + 0.06 * (df.mean_temp_c - 20).abs())
df["failed_90d"] = (rng.random(n) < 1 / (1 + np.exp(-logit))).astype(int)
print(df.failed_90d.mean())            # about 0.2: one in five units fails

### A pipeline that cannot leak

Numbers need scaling, categories need one-hot encoding, and both must be fitted on training data only. `ColumnTransformer` inside a `Pipeline` handles this, and cross-validation then refits the preprocessing on each fold automatically.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

num = ["age_days", "mean_temp_c", "humidity_pct", "power_cycles"]
cat = ["site", "firmware"]
X, y = df[num + cat], df["failed_90d"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)

pre = ColumnTransformer([
    ("num", StandardScaler(), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])
models = {
    "logistic": LogisticRegression(max_iter=2000),
    "random_forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=5, random_state=0, n_jobs=-1),
    "grad_boost": HistGradientBoostingClassifier(learning_rate=0.05, max_iter=300, random_state=0),
}
cv = StratifiedKFold(5, shuffle=True, random_state=0)
for name, model in models.items():
    pipe = Pipeline([("prep", pre), ("model", model)])
    s = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="roc_auc")
    print(f"{name:<14} ROC-AUC {s.mean():.3f} ± {s.std():.3f}")

```text
logistic       ROC-AUC 0.742 ± 0.027
random_forest  ROC-AUC 0.733 ± 0.020
grad_boost     ROC-AUC 0.694 ± 0.033
```

### Which features matter?

**Permutation importance** shuffles one feature at a time on held-out data and measures how much the score drops. It works for any model and reflects what the model actually relies on.

In [ ]:
from sklearn.inspection import permutation_importance

best = Pipeline([("prep", pre), ("model", LogisticRegression(max_iter=2000))]).fit(X_tr, y_tr)
imp = permutation_importance(best, X_te, y_te, scoring="roc_auc", n_repeats=10, random_state=0)
print(pd.Series(imp.importances_mean, index=X.columns).sort_values(ascending=False).round(3))

### Hands-on: engineer, tune, decide

1. Add a feature `temp_dev = abs(mean_temp_c - 20)` and rerun the logistic regression cross-validation. Does ROC-AUC improve?
2. Tune the gradient-boosting model: try `max_depth=3`, `min_samples_leaf=40`, `l2_regularization=1.0`, `learning_rate=0.03`. How close does it get?
3. Technicians can visit 100 units this quarter. Using the best model's `predict_proba` on the test set, how many of the 100 highest-risk units actually failed? Compare with picking 100 at random.

#### Solution: a solution

In [ ]:
df["temp_dev"] = (df.mean_temp_c - 20).abs()
num2 = num + ["temp_dev"]
X2 = df[num2 + cat]
X2_tr, X2_te, y_tr, y_te = train_test_split(X2, y, test_size=0.2, stratify=y, random_state=0)
pre2 = ColumnTransformer([("num", StandardScaler(), num2),
                          ("cat", OneHotEncoder(handle_unknown="ignore"), cat)])
logit_pipe = Pipeline([("prep", pre2), ("model", LogisticRegression(max_iter=2000))])
print(cross_val_score(logit_pipe, X2_tr, y_tr, cv=cv, scoring="roc_auc").mean())   # about 0.752

gb = HistGradientBoostingClassifier(learning_rate=0.03, max_iter=300, max_depth=3,
                                    min_samples_leaf=40, l2_regularization=1.0, random_state=0)
print(cross_val_score(Pipeline([("prep", pre), ("model", gb)]), X_tr, y_tr,
                      cv=cv, scoring="roc_auc").mean())                         # about 0.73

logit_pipe.fit(X2_tr, y_tr)
risk = logit_pipe.predict_proba(X2_te)[:, 1]
top100 = np.argsort(risk)[::-1][:100]
print("failures in top 100:", y_te.iloc[top100].sum())
print("expected at random:", round(100 * y_te.mean()))

The engineered feature lifts the linear model, and regularising the boosted model closes most of its gap. The top-100 question is the one the business cares about: a ranked list that roughly doubles the hit rate of random visits is worth real money, whatever the AUC says. Always translate metrics into the decision they support.

## Check yourself

Take the quiz at the end of [the lesson page](../../lessons/04-classical-ml.html). Then close everything and write three or four sentences in your learning log about what you learned, from memory.